# Phase 2 — Feature Engineering Exploration

This notebook runs the full Phase 2 pipeline and provides visual sanity checks:
- Calendar feature distributions
- Correlation matrix of temporal features vs targets
- Weather-derived feature time series
- NaN audit
- Feature importance preview (mutual information)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

PROCESSED = Path('../data/processed')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
from src.features.pipeline import build_features, get_feature_cols, TARGET_COLS

# Run the full Phase 2 pipeline (reads Phase 1 master.parquet)
df = build_features(PROCESSED, force=False)
print(f'Shape: {df.shape}')
df.head(3)

## 1. NaN audit

In [ ]:
nan_pct = df.isna().mean().mul(100).sort_values(ascending=False)
nan_pct = nan_pct[nan_pct > 0]
print(f'{len(nan_pct)} columns with missing values (out of {df.shape[1]}):\n')
print(nan_pct.to_string())

## 2. Target time series overview

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, col in zip(axes, TARGET_COLS):
    # Weekly resample for clarity
    df[col].resample('W').mean().plot(ax=ax, lw=1)
    ax.set_ylabel('MW (weekly avg)')
    ax.set_title(col)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../results/target_overview.png', bbox_inches='tight')
plt.show()

## 3. Diurnal profiles (by season)

In [ ]:
season_labels = {0: 'Winter', 1: 'Spring', 2: 'Summer', 3: 'Autumn'}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)

for ax, target in zip(axes.flat, TARGET_COLS):
    for season_id, season_name in season_labels.items():
        subset = df[df['season'] == season_id]
        profile = subset.groupby('hour')[target].mean()
        ax.plot(profile.index, profile.values, label=season_name)
    ax.set_title(target)
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('MW')
    ax.legend(fontsize=8)

plt.suptitle('Diurnal profiles by season', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/diurnal_profiles.png', bbox_inches='tight')
plt.show()

## 4. Correlation of lag features with targets

In [ ]:
lag_cols = [c for c in df.columns if '_lag' in c or '_roll' in c]
corr_matrix = df[TARGET_COLS + lag_cols[:20]].corr()[TARGET_COLS].drop(TARGET_COLS)

fig, ax = plt.subplots(figsize=(8, max(6, len(corr_matrix) * 0.3)))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7}
)
ax.set_title('Pearson correlation: lag/rolling features → targets')
plt.tight_layout()
plt.savefig('../results/lag_correlation.png', bbox_inches='tight')
plt.show()

## 5. Weather-derived features vs solar output

In [ ]:
# Pick a sunny summer week for illustration
sample = df.loc['2019-07-01':'2019-07-07']

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

ax1.plot(sample.index, sample['solar_mw'], color='gold', label='Solar MW')
ax1.set_ylabel('Solar MW'); ax1.legend()

ax2.plot(sample.index, sample['clearsky_index'], color='orange', label='Clearsky index')
ax2.set_ylabel('Clearsky fraction'); ax2.legend()

ax3.plot(sample.index, sample['de_temp'], color='red', label='Temp °C')
ax3.plot(sample.index, sample['hdd'], color='blue', label='HDD', alpha=0.5)
ax3.set_ylabel('°C'); ax3.legend()

plt.suptitle('Weather-derived features — sample week 2019-07-01', fontsize=12)
plt.tight_layout()
plt.savefig('../results/weather_features_sample.png', bbox_inches='tight')
plt.show()

## 6. Mutual information: feature importance preview

In [ ]:
from sklearn.feature_selection import mutual_info_regression

target = 'load_mw'
feature_cols = get_feature_cols(df)

# Use a sample to keep MI fast
sample = df[feature_cols + [target]].dropna().sample(5000, random_state=42)
X = sample[feature_cols]
y = sample[target]

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

top30 = mi_series.head(30)
fig, ax = plt.subplots(figsize=(8, 9))
top30.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Mutual information score')
ax.set_title(f'Top 30 features for target: {target}')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/feature_importance_mi.png', bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
print(mi_series.head(10).to_string())

## 7. Train/val/test split preview

In [ ]:
from src.features.scaling import split_and_scale

splits = split_and_scale(
    df,
    target_cols=TARGET_COLS,
    feature_cols=get_feature_cols(df),
    train_end='2021-12-31',
    val_end='2022-12-31',
)

for split_name in ['train', 'val', 'test']:
    s = splits[split_name]
    print(f"{split_name:6s}: {len(s):>6,} rows  "
          f"({s.index.min().date()} → {s.index.max().date()})")